# Demo 1 — Phoenix RAG Evaluations (phoenix-evals 3.0.0)

Runs post-hoc evaluations on the RAG pipeline using **Ollama as the judge LLM** (LLM-as-a-judge pattern).

Three evaluation dimensions:

| Evaluator | What it checks | phoenix-evals 3.x class |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context? | `FaithfulnessEvaluator` |
| **Correctness** | Is the answer correct vs a reference answer? | `CorrectnessEvaluator` |
| **Retrieval Relevance** | Is the retrieved context relevant to the question? | `DocumentRelevanceEvaluator` |

**Prerequisites:** `docker compose up -d` in `demo1-phoenix-rag/`, models pulled, notebook `01_ingest_and_embed.ipynb` run first.

In [6]:
import os, sys
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")

PHOENIX_BASE_URL = os.getenv("PHOENIX_BASE_URL", "http://localhost:6006")
OLLAMA_BASE_URL  = os.getenv("OLLAMA_BASE_URL",  "http://localhost:11434")
OLLAMA_LLM_MODEL = os.getenv("OLLAMA_LLM_MODEL", "llama3.2")

# Add app/ to path so we can import the RAG pipeline directly
sys.path.insert(0, "..")

print(f"Phoenix : {PHOENIX_BASE_URL}")
print(f"Ollama  : {OLLAMA_BASE_URL}  model={OLLAMA_LLM_MODEL}")

Phoenix : http://localhost:6006
Ollama  : http://localhost:11434  model=llama3.2


## 1. Build Evaluation Dataset

Run 5 representative tax questions through the RAG pipeline.
Each call is automatically traced to Phoenix (LangChain instrumentation fires on import).

The pipeline returns `{"answer": ..., "context_chunks": [...]}` so we capture all three
columns needed for evaluation: **question**, **context**, **answer**.

In [7]:
from app.rag_pipeline import query_rag

# Questions + known reference answers (used for CorrectnessEvaluator)
QA_PAIRS = [
    (
        "What is the standard deduction for a single filer in 2024?",
        "$14,600",
    ),
    (
        "How are long-term capital gains taxed for someone earning $100,000?",
        "15% rate applies for most middle-income earners",
    ),
    (
        "What is the 401(k) contribution limit for 2024 including catch-up?",
        "$30,500 total with catch-up for age 50+",
    ),
    (
        "Can I deduct student loan interest if I earn $80,000 as a single filer?",
        "Yes, up to $2,500 but phases out between $75,000–$90,000 MAGI",
    ),
    (
        "What are the Roth IRA income phase-out ranges for single filers in 2024?",
        "$146,000–$161,000 for single filers",
    ),
]

rows = []
for question, reference in QA_PAIRS:
    print(f"Querying: {question[:60]}...")
    result = query_rag(question)
    rows.append({
        "question":   question,
        "context":    "\n\n".join(result.get("context_chunks", [])),
        "answer":     result["answer"],
        "reference":  reference,
    })

eval_df = pd.DataFrame(rows)
print(f"\nEval dataset ready: {len(eval_df)} rows")
eval_df[["question", "answer"]].head()

Querying: What is the standard deduction for a single filer in 2024?...
Querying: How are long-term capital gains taxed for someone earning $1...
Querying: What is the 401(k) contribution limit for 2024 including cat...
Querying: Can I deduct student loan interest if I earn $80,000 as a si...
Querying: What are the Roth IRA income phase-out ranges for single fil...

Eval dataset ready: 5 rows


,question,answer
0,What is the standard deduction for a single fi...,"According to the context, the standard deducti..."
1,How are long-term capital gains taxed for some...,"Based on the provided context, I can answer yo..."
2,What is the 401(k) contribution limit for 2024...,"According to the provided context excerpt, the..."
3,Can I deduct student loan interest if I earn $...,"According to the context excerpt, yes, you can..."
4,What are the Roth IRA income phase-out ranges ...,According to the context excerpt from the reti...


In [8]:
# Inspect the context column so we can see what the retriever pulled
eval_df[["question", "context"]].assign(
    context_preview=eval_df["context"].str[:200]
)[["question", "context_preview"]]

,question,context_preview
0,What is the standard deduction for a single fi...,STANDARD DEDUCTION AND ITEMIZED DEDUCTIONS — T...
1,How are long-term capital gains taxed for some...,SHORT-TERM CAPITAL GAINS\n\nAssets held for on...
2,What is the 401(k) contribution limit for 2024...,"SIMPLE IRA: Employee contribution limit $16,00..."
3,Can I deduct student loan interest if I earn $...,ABOVE-THE-LINE DEDUCTIONS (ADJUSTMENTS TO INCO...
4,What are the Roth IRA income phase-out ranges ...,2024 Long-Term Capital Gains Tax Rates:\n- 0%:...


## 2. Set Up the Judge LLM

phoenix-evals 3.x uses a three-layer adapter pattern:

```
ChatOllama  →  LangChainModelAdapter  →  LLM (phoenix judge wrapper)
```

`ChatOllama` is already available from `langchain-ollama` installed in this venv.

In [9]:
from langchain_ollama import ChatOllama
from phoenix.evals.llm.adapters.langchain.adapter import LangChainModelAdapter
from phoenix.evals.llm.registries import PROVIDER_REGISTRY
from phoenix.evals import LLM

# arize-phoenix-evals 3.x removed LLM(client=adapter); register Ollama as a custom provider
def _create_ollama_langchain_client(model: str, is_async: bool = False, **kwargs):
    return ChatOllama(model=model, **kwargs)

PROVIDER_REGISTRY.register_provider(
    provider="ollama",
    adapter_class=LangChainModelAdapter,
    client_factory=_create_ollama_langchain_client,
    dependencies=["langchain-ollama"],
)

judge_llm = LLM(provider="ollama", model=OLLAMA_LLM_MODEL, base_url=OLLAMA_BASE_URL)

# Smoke-test: make sure Ollama is reachable
test = ChatOllama(model=OLLAMA_LLM_MODEL, base_url=OLLAMA_BASE_URL).invoke("Reply with only the word: ready")
print(f"Judge LLM smoke test → {test.content.strip()}")

Judge LLM smoke test → ready


## 3. Faithfulness Evaluation (Hallucination Detection)

`FaithfulnessEvaluator` asks the judge: *"Is every claim in the answer supported by the context?"*

**Input columns required:** `input` (question), `context` (retrieved context), `output` (answer)

Returns a label of `faithful` or `unfaithful` plus a score (1.0 / 0.0).

In [10]:
from phoenix.evals.metrics import FaithfulnessEvaluator
from phoenix.evals import evaluate_dataframe

faithfulness_df = eval_df.rename(columns={
    "question": "input",
    "context":  "context",
    "answer":   "output",
})

faithfulness_results = evaluate_dataframe(
    faithfulness_df[["input", "context", "output"]],
    evaluators=[FaithfulnessEvaluator(judge_llm)],
)

faithfulness_labels = faithfulness_results["faithfulness_score"].apply(
    lambda x: x.get("label") if isinstance(x, dict) else None
)
print("Faithfulness evaluation complete")
print(faithfulness_labels.value_counts().to_string())
faithfulness_results

/var/folders/zp/3gl4bcp52_v07zwqx9qrf3gm0000gq/T/ipykernel_28910/1829033666.py:10: DeprecationWarning: Positional arguments for evaluate_dataframe are deprecated and will be removed in a future version. Please use keyword arguments instead.
  faithfulness_results = evaluate_dataframe(


Evaluating Dataframe |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

Faithfulness evaluation complete
faithfulness_score
unfaithful    4
faithful      1


,input,context,output,faithfulness_execution_details,faithfulness_score
0,What is the standard deduction for a single fi...,STANDARD DEDUCTION AND ITEMIZED DEDUCTIONS — T...,"According to the context, the standard deducti...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 0.0, 'label'..."
1,How are long-term capital gains taxed for some...,SHORT-TERM CAPITAL GAINS\n\nAssets held for on...,"Based on the provided context, I can answer yo...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 0.0, 'label'..."
2,What is the 401(k) contribution limit for 2024...,"SIMPLE IRA: Employee contribution limit $16,00...","According to the provided context excerpt, the...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 0.0, 'label'..."
3,Can I deduct student loan interest if I earn $...,ABOVE-THE-LINE DEDUCTIONS (ADJUSTMENTS TO INCO...,"According to the context excerpt, yes, you can...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'..."
4,What are the Roth IRA income phase-out ranges ...,2024 Long-Term Capital Gains Tax Rates:\n- 0%:...,According to the context excerpt from the reti...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 0.0, 'label'..."


## 4. Correctness Evaluation (QA Accuracy)

`CorrectnessEvaluator` asks the judge: *"Is the answer factually correct?"*

**Input columns required:** `input` (question), `output` (generated answer)

Returns `correct` / `incorrect`.

In [11]:
from phoenix.evals.metrics import CorrectnessEvaluator

correctness_df = eval_df.rename(columns={
    "question": "input",
    "answer":   "output",
})

correctness_results = evaluate_dataframe(
    correctness_df[["input", "output"]],
    evaluators=[CorrectnessEvaluator(judge_llm)],
)

correctness_labels = correctness_results["correctness_score"].apply(
    lambda x: x.get("label") if isinstance(x, dict) else None
)
print("Correctness evaluation complete")
print(correctness_labels.value_counts().to_string())
correctness_results

/var/folders/zp/3gl4bcp52_v07zwqx9qrf3gm0000gq/T/ipykernel_28910/2147472778.py:8: DeprecationWarning: Positional arguments for evaluate_dataframe are deprecated and will be removed in a future version. Please use keyword arguments instead.
  correctness_results = evaluate_dataframe(


Evaluating Dataframe |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

Correctness evaluation complete
correctness_score
incorrect    5


,input,output,correctness_execution_details,correctness_score
0,What is the standard deduction for a single fi...,"According to the context, the standard deducti...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
1,How are long-term capital gains taxed for some...,"Based on the provided context, I can answer yo...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
2,What is the 401(k) contribution limit for 2024...,"According to the provided context excerpt, the...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
3,Can I deduct student loan interest if I earn $...,"According to the context excerpt, yes, you can...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
4,What are the Roth IRA income phase-out ranges ...,According to the context excerpt from the reti...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."


## 5. Document Relevance Evaluation (Retrieval Quality)

`DocumentRelevanceEvaluator` asks the judge: *"Is the retrieved context relevant to the question?"*

This catches retriever failures — cases where the vector search returns off-topic chunks.

**Input columns required:** `input` (question), `document_text` (retrieved context)

Returns `relevant` / `irrelevant`.

In [12]:
from phoenix.evals.metrics import DocumentRelevanceEvaluator

relevance_df = eval_df.rename(columns={
    "question": "input",
    "context":  "document_text",
})

relevance_results = evaluate_dataframe(
    relevance_df[["input", "document_text"]],
    evaluators=[DocumentRelevanceEvaluator(judge_llm)],
)

relevance_labels = relevance_results["document_relevance_score"].apply(
    lambda x: x.get("label") if isinstance(x, dict) else None
)
print("Document relevance evaluation complete")
print(relevance_labels.value_counts().to_string())
relevance_results

/var/folders/zp/3gl4bcp52_v07zwqx9qrf3gm0000gq/T/ipykernel_28910/67053819.py:8: DeprecationWarning: Positional arguments for evaluate_dataframe are deprecated and will be removed in a future version. Please use keyword arguments instead.
  relevance_results = evaluate_dataframe(


Evaluating Dataframe |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

Document relevance evaluation complete
document_relevance_score
relevant    5


,input,document_text,document_relevance_execution_details,document_relevance_score
0,What is the standard deduction for a single fi...,STANDARD DEDUCTION AND ITEMIZED DEDUCTIONS — T...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'document_relevance', 'score': 1.0, '..."
1,How are long-term capital gains taxed for some...,SHORT-TERM CAPITAL GAINS\n\nAssets held for on...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'document_relevance', 'score': 1.0, '..."
2,What is the 401(k) contribution limit for 2024...,"SIMPLE IRA: Employee contribution limit $16,00...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'document_relevance', 'score': 1.0, '..."
3,Can I deduct student loan interest if I earn $...,ABOVE-THE-LINE DEDUCTIONS (ADJUSTMENTS TO INCO...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'document_relevance', 'score': 1.0, '..."
4,What are the Roth IRA income phase-out ranges ...,2024 Long-Term Capital Gains Tax Rates:\n- 0%:...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'document_relevance', 'score': 1.0, '..."


## 6. Combined Summary Dashboard

Merge all three eval dimensions into one DataFrame for at-a-glance quality assessment.

In [13]:
summary = eval_df[["question", "answer"]].copy()
summary["faithful"]         = faithfulness_labels.values
summary["correct"]          = correctness_labels.values
summary["context_relevant"] = relevance_labels.values

print("=" * 50)
print("RAG Evaluation Summary")
print("=" * 50)
print(f"Queries evaluated   : {len(summary)}")
print(f"Faithful (no halluc): {(summary['faithful'] == 'faithful').mean():.0%}")
print(f"Correct answers     : {(summary['correct'] == 'correct').mean():.0%}")
print(f"Relevant context    : {(summary['context_relevant'] == 'relevant').mean():.0%}")
print()
summary

RAG Evaluation Summary
Queries evaluated   : 5
Faithful (no halluc): 20%
Correct answers     : 0%
Relevant context    : 100%



,question,answer,faithful,correct,context_relevant
0,What is the standard deduction for a single fi...,"According to the context, the standard deducti...",unfaithful,incorrect,relevant
1,How are long-term capital gains taxed for some...,"Based on the provided context, I can answer yo...",unfaithful,incorrect,relevant
2,What is the 401(k) contribution limit for 2024...,"According to the provided context excerpt, the...",unfaithful,incorrect,relevant
3,Can I deduct student loan interest if I earn $...,"According to the context excerpt, yes, you can...",faithful,incorrect,relevant
4,What are the Roth IRA income phase-out ranges ...,According to the context excerpt from the reti...,unfaithful,incorrect,relevant


## 7. View Traces in Phoenix

All 5 queries above were automatically traced. Open Phoenix to inspect:
- Each RAG span (retrieval + generation)
- Token counts, latency, and context passed to the LLM

```
http://localhost:6006
```

Look for the **tax-rag-demo** project → click any trace to see:
- `rag_query` root span
- `retrieval` child span with per-chunk attributes
- `generation` child span with prompt + completion

## 8. Log Evaluations Back to Phoenix (REST API)

Phoenix accepts evaluation annotations via `POST /v1/span_evaluations`.
To log evals to the exact spans, we need the span IDs — which come from the OTel context
captured during the `query_rag()` calls above.

The cells below show how to pull span IDs from Phoenix and attach eval results.

```python
import requests

# 1. Pull recent spans for this project
resp = requests.get(
    f"{PHOENIX_BASE_URL}/v1/projects",
    headers={"Accept": "application/json"},
)
projects = resp.json()
print(projects)

# 2. Once you have span_ids, post evaluations:
# POST /v1/span_evaluations
# Body: {"data": [{"span_id": "...", "name": "faithfulness", "label": "faithful", "score": 1.0}]}
```

> **For the demo recording:** The eval results displayed above (the summary DataFrame) are the
> key output to show. Phoenix's Evaluations tab auto-populates when evaluations are logged via
> the full `arize-phoenix` server package — available if you run Phoenix as a local server
> (`pip install arize-phoenix`) instead of the Docker image.